# Cell 1: About this notebook

Stage 3 of switchgear-estimation-assistant. Independent personal project, synthetic data only.

This notebook runs LLM extraction (`gemma4:e2b` through Ollama, structured output) on the files the
**laptop** built: the showcase, the 20 test projects (T01 to T20) and the 20 reworded projects
(R01 to R20, seeds 2001 to 2020). It saves one recording per file for the app's recorded mode.

It never rebuilds a data set. Windows and Linux builds of the same seed differ in their compressed
bytes, and a recording is accepted only for the exact bytes it was made from (SHA-256). On the laptop, run:

```powershell
.venv\Scripts\python.exe -m backend.record pack --out laptop-files.zip
```

and upload `laptop-files.zip` in Cell 8. Runtime: a T4 GPU (Runtime > Change runtime type).

**This notebook has not been run yet.** It has no saved outputs.

In [ ]:
# Cell 2: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 3: Install zstd (unpacking) and pciutils + lshw (GPU detection), needed by the Ollama installer
!apt-get install -y -q zstd pciutils lshw

In [ ]:
# Cell 4: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Cell 5: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8
!ollama --version

In [ ]:
# Cell 6: Download the model and show its digest (saved in every recording as the model version)
!ollama pull gemma4:e2b
!ollama list

In [ ]:
# Cell 7: Get the code from the uploaded repo.zip and install the app requirements
%cd /content
from google.colab import files
uploaded = files.upload()
!mkdir -p /content/switchgear-estimation-assistant
!unzip -q -o repo.zip -d /content/switchgear-estimation-assistant
%cd /content/switchgear-estimation-assistant
!pip install -q -r requirements.txt

In [ ]:
# Cell 8: Upload laptop-files.zip (made on the laptop by: python -m backend.record pack)
from google.colab import files

uploaded = files.upload()
assert "laptop-files.zip" in uploaded, "upload the file named laptop-files.zip"
!ls -l laptop-files.zip

In [ ]:
# Cell 9: Show what the laptop built: environment and number of files per set
import collections
import json
import zipfile

with zipfile.ZipFile("laptop-files.zip") as z:
    manifest = json.loads(z.read("manifest.json"))
print(json.dumps(manifest["built_on"], indent=2))
print(collections.Counter(entry["set"] for entry in manifest["files"]))

In [ ]:
# Cell 10: Mount Google Drive and create the recordings folder (kept if the Colab session stops)
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/switchgear-recordings
!ls -la /content/drive/MyDrive/switchgear-recordings

In [ ]:
# Cell 11: Run LLM extraction on every file and save the recordings in Google Drive
# Every file is checked against its SHA-256 in the manifest first; a file with other bytes is refused.
# --resume skips files already recorded in the Drive folder (same bytes, same prompt version). After an
# interruption, run the cells again from the top: finished files are skipped, the rest are recorded.
import os

os.environ["PYTHONPATH"] = "ai_engine:."
!PYTHONPATH=ai_engine:. python -m backend.record run --zip laptop-files.zip --mode ollama --out /content/drive/MyDrive/switchgear-recordings --resume

In [ ]:
# Cell 12: Summary: files recorded, model calls and seconds per document (read from Google Drive)
import json
from pathlib import Path

RECORDINGS = Path("/content/drive/MyDrive/switchgear-recordings")
rows = []
for path in sorted(RECORDINGS.rglob("*.json")):
    rec = json.loads(path.read_text())
    rows.append((str(path.relative_to(RECORDINGS)), rec["model"], rec["model_version"], len(rec["calls"]),
                 round(sum(c["seconds"] for c in rec["calls"]), 1)))
for row in rows:
    print(*row, sep="  ")
print(len(rows), "recordings;", round(sum(r[4] for r in rows) / max(1, len(rows)), 1), "s per document on average")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# Cell 13: Download the recordings from Google Drive; unzip them into data\recorded\ on the laptop
# The zip holds <set>/<project>/<file>.json, the layout of data\recorded\.
!rm -f /content/recordings.zip
!cd /content/drive/MyDrive/switchgear-recordings && zip -q -r /content/recordings.zip .
from google.colab import files

files.download("/content/recordings.zip")